# Projeto 1 — Hello CUDA no Google Colab

Este notebook compila e executa o projeto `hello-cuda` utilizando uma GPU NVIDIA disponibilizada pelo Google Colab.

Antes de executar:

1. Abra **Ambiente de execução → Alterar tipo de ambiente de execução**.
2. Em **Acelerador de hardware**, selecione uma opção de **GPU**.
3. Salve a configuração.
4. Execute as células na ordem.


## 1. Verificar a GPU e o compilador CUDA

In [ ]:
!nvidia-smi
print("\n--- Compilador CUDA ---\n")
!nvcc --version

A primeira saída identifica a GPU atribuída ao ambiente. A segunda confirma que o compilador `nvcc` está disponível.

## 2. Enviar o arquivo `hello-cuda.zip`

Clique em **Escolher arquivos** e selecione o ZIP do projeto.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Arquivos enviados:", list(uploaded.keys()))

## 3. Extrair o projeto

In [ ]:
from pathlib import Path
import os
import zipfile

zip_files = list(Path('/content').glob('*.zip'))
if not zip_files:
    raise FileNotFoundError('Nenhum arquivo ZIP foi enviado.')

zip_path = zip_files[0]
print('Extraindo:', zip_path.name)

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall('/content')

project_dir = Path('/content/hello-cuda')
if not project_dir.exists():
    raise FileNotFoundError(
        'A pasta /content/hello-cuda não foi encontrada dentro do ZIP.'
    )

os.chdir(project_dir)
print('Diretório atual:', Path.cwd())
print('\nArquivos do projeto:')
for path in sorted(project_dir.rglob('*')):
    if path.is_file():
        print('-', path.relative_to(project_dir))

## 4. Compilar com `nvcc`

O arquivo `.cu` contém código executado na CPU e um kernel executado na GPU.

In [ ]:
!nvcc -std=c++17 -O2 -lineinfo -Iinclude src/main.cu -o hello_cuda

## 5. Executar com a configuração padrão

A configuração padrão lança dois blocos com quatro threads em cada bloco:

In [ ]:
!./hello_cuda

A chamada equivalente no código é:

```cpp
helloCudaKernel<<<2, 4>>>();
```

O total é:

$$2\ \text{blocos} \times 4\ \text{threads por bloco} = 8\ \text{threads}$$

O índice global unidimensional é calculado por:

$$i = \text{blockIdx.x}\times\text{blockDim.x}+\text{threadIdx.x}$$

## 6. Fazer experimentos

A sintaxe do programa é:

```text
./hello_cuda [blocos] [threads_por_bloco] [dispositivo]
```

In [ ]:
print('Experimento: 1 bloco e 8 threads')
!./hello_cuda 1 8

print('\nExperimento: 3 blocos e 5 threads')
!./hello_cuda 3 5

## 7. Exercício guiado

Antes de executar a célula abaixo, responda:

- Quantas threads serão criadas?
- Qual será o maior índice global?
- Quantas threads terão `threadIdx.x == 0`?
- Quais índices globais pertencem ao bloco 2?

In [ ]:
!./hello_cuda 4 6

### Resposta do exercício

- Total: $4\times6=24$ threads.
- Maior índice global: $23$.
- Quatro threads possuem `threadIdx.x == 0`, uma por bloco.
- No bloco 2, os índices globais vão de $12$ até $17$.

> A ordem das linhas impressas pela GPU pode mudar entre execuções. Isso não altera os índices calculados.